# Load data

### Julia (Need to be ran in terminal before start (for PubMedRAG))
### Install Julia
JULIA_VERSION=1.10.4
cd $HOME
wget https://julialang-s3.julialang.org/bin/linux/x64/1.10/julia-$JULIA_VERSION-linux-x86_64.tar.gz
tar -xvzf julia-$JULIA_VERSION-linux-x86_64.tar.gz
ln -s julia-$JULIA_VERSION julia

export PATH="$HOME/julia/bin:$PATH" # add to PATH (put this in ~/.bashrc or your sbatch script)
julia --version
#### Run Julia
!bash slurm/run_julia.sh

In [ ]:

import json
import os
from explain.util import load_data
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from explain.llm.data_generator import DataGenerator
from explain.util import set_perturbation_ner_mapping
from explain.eval.score.evaluate import evaluate
from explain.kg.starkprimekg_utils import get_kg_info
from explain.kg.starkprimekg import StarkPrimeKG
from explain.literature.harmonizome_utils import get_harmonizome_info
from explain.literature.wikipedia_utils import get_wikipedia_info
from explain.literature.paperqa_utils import get_paperqa_info
from script.generate import fetch_tool_info
# pubmed_info can be imported after running the above commands
# from explain.literature.pubmed_utils import get_pubmed_info

<All keys matched successfully>


In [15]:
os.chdir('/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain')

In [17]:
# for other dataset (Tahoe, RXRX, etc.) add corresponding index_tag 
index_tag = ""
pert_path = 'data/perturbation_ner_mapping.json'
perturbation_ner_mapping = json.load(open(pert_path, 'r'))
perturbation_ner_mapping_data_indices = [index_tag + str(item['index']) for item in perturbation_ner_mapping]

In [33]:
set_perturbation_ner_mapping(perturbation_ner_mapping, perturbation_ner_mapping_data_indices)

In [21]:
report_list = []
structure_explain_list = []
tool_list = ['kg-ner-subgraph', 'harmonizome', 'wikipedia']   
data_generator = DataGenerator(model_name='anthropic', tool_list=tool_list, pert_path=pert_path)
perturbations = data_generator.perturbation_cell_context


2025-09-08 13:42:24.830 | INFO     | explain.llm._client:__init__:695 - Initialized unified LLM client with provider: anthropic
2025-09-08 13:42:24.861 | INFO     | explain.llm._client:__init__:695 - Initialized unified LLM client with provider: anthropic
2025-09-08 13:42:24.893 | INFO     | explain.llm._client:__init__:695 - Initialized unified LLM client with provider: anthropic


In [ ]:
kg = StarkPrimeKG()

Loading embeddings from /rxrx/data/user/hamed.shirzad/outgoing/stark_prime_kg/pritamdeka/S-PubMedBERT-MS-MARCO/node_embeddings.pt


# Data generation

In [38]:
report_list = []
structure_explain_list = []

for i, perturbation in enumerate(tqdm(perturbations[:3])):
        index = index_tag + str(perturbation['index'])
        additional_info = ""
        kg_info = ""
        harmonizome_info = ""
        wikipedia_info = ""
        extracted_graph_info = {}
        papers_info = {}
        perturbation_text = json.dumps(perturbation, indent=4)
        question = "**Q: How does the following perturbation influence the cell in the described context, mechanistically and functionally?**\n\n"
        question += perturbation_text
        result = {'additional': ''}
        for tool_name in tool_list:
            # Load KG information
            if 'kg' in tool_name:
                kg_info, extracted_graph_info = get_kg_info(index, perturbation, tool_name, kg, False, 1)
                kg_info = data_generator.post_process_additional_info(kg_info, tool_name, perturbation)
                result["kg_info"] = kg_info
                result["extracted_graph_info"] = extracted_graph_info
                result['additional'] += kg_info
            # Load harmonizome information
            elif 'harmonizome' in tool_name:
                ner_flag = 'ner' in tool_name
                harmonizome_info = get_harmonizome_info(index, perturbation, is_ner=ner_flag)
                harmonizome_info = data_generator.post_process_additional_info(harmonizome_info, tool_name, perturbation)
                result["harmonizome_info"] = harmonizome_info
                result['additional'] += harmonizome_info
            # Load wikipedia information
            elif 'wikipedia' in tool_name:
                wikipedia_info = get_wikipedia_info(index, perturbation)
                wikipedia_info = data_generator.post_process_additional_info(wikipedia_info, tool_name, perturbation)
                result["wikipedia_info"] = wikipedia_info
                result['additional'] += wikipedia_info
        # report generation
        additional_info += result['additional']
        report = data_generator.generate_report(perturbation, additional_info)
        report_dict = {'index': index, 'perturbation': perturbation, 'report_text': report, 'question': question,
                'kg_info': kg_info, 'extracted_graph_info': extracted_graph_info, 'harmonizome_info': harmonizome_info,
                'wikipedia_info': wikipedia_info}
        report_list.append(report_dict)
        # structure explain generation
        structure_explain = data_generator.generate_structure_explain(report, question)
        thinking, answer, explain, dag = data_generator.process_structure_explain(structure_explain)
        structure_explain_dict = {'index': index, 'input_perturbation': perturbation, 'thinking': thinking,
        'answer': answer, 'explain': explain, 'dag': dag, 'raw_response': structure_explain,
        'question': question, 'input_report_text': report, 'additional_info': additional_info}
        structure_explain_list.append(structure_explain_dict)

 67%|██████▋   | 2/3 [02:41<01:21, 81.29s/it]

DrugBankSearcher initialized successfully


100%|██████████| 3/3 [03:55<00:00, 78.66s/it]


# Data

In [42]:
import pandas as pd

report_df = pd.DataFrame(report_list)
structure_explain_df = pd.DataFrame(structure_explain_list)

In [43]:
report_df

,index,perturbation,report_text,question,kg_info,extracted_graph_info,harmonizome_info,wikipedia_info
0,0,"{'index': 0, 'perturbation': {'context': {'per...",# Mechanistic Report: Bevacizumab Effects in A...,**Q: How does the following perturbation influ...,## KNOWLEDGE GRAPH INFORMATION\nThe retrieved ...,"{'node_list': ['749', '16510'], 'subgraph': [3...","## GENE INFORMATION\n(""### TARGET GENE INFORMA...",## WIKIPEDIA INFORMATION\n## Bevacizumab\nBeva...
1,1,"{'index': 1, 'perturbation': {'context': {'per...",# Mechanistic Report: Nintedanib Inhibition of...,**Q: How does the following perturbation influ...,## KNOWLEDGE GRAPH INFORMATION\nThe retrieved ...,"{'node_list': ['749', '14413'], 'subgraph': [1...","## GENE INFORMATION\n(""### TARGET GENE INFORMA...",## WIKIPEDIA INFORMATION\n## Nintedanib\nNinte...
2,2,"{'index': 2, 'perturbation': {'context': {'per...",# Biomedical Reasoning Assistant – Mechanistic...,**Q: How does the following perturbation influ...,## KNOWLEDGE GRAPH INFORMATION\nThe retrieved ...,"{'node_list': ['21763', '749'], 'subgraph': [3...","## GENE INFORMATION\n(""### TARGET GENE INFORMA...",## WIKIPEDIA INFORMATION\n## VEGF\nVascular en...


In [46]:
print(report_df.iloc[0]['kg_info'])

## KNOWLEDGE GRAPH INFORMATION
The retrieved subgraph contains 2 nodes and 2 edges.

Nodes:
- Node ID: 749, Doc Info: - name: VEGFA - type: gene/protein - source: NCBI - details:   - query: VEGFA   - alias (other gene names): ['L-VEGF', 'MVCD1', 'VEGF', 'VPF']   - genomic_pos (genomic position): {'chr': '6', 'end': 43786487, 'ensemblgene': 'ENSG00000112715', 'start': 43770184, 'strand': 1}   - name (gene name): vascular endothelial growth factor A   - summary (protein summary text): This gene is a member of the PDGF/VEGF growth factor family. It encodes a heparin-binding protein, which exists as a disulfide-linked homodimer. This growth factor induces proliferation and migration of vascular endothelial cells, and is essential for both physiological and pathological angiogenesis. Disruption of this gene in mice resulted in abnormal embryonic blood vessel formation. This gene is upregulated in many known tumors and its expression is correlated with tumor stage and progression. Elevated l

In [47]:
print(report_df.iloc[0]['report_text'])

# Mechanistic Report: Bevacizumab Effects in Angiogenic/Tumor Context

## 1. Perturbation Description

**Bevacizumab** is a humanized monoclonal antibody (IgG1) that functions as a VEGF (Vascular Endothelial Growth Factor) inhibitor. Key characteristics:

- **Type**: Therapeutic monoclonal antibody (chemical perturbation via biological agent)
- **Primary Target**: Circulating VEGF-A protein (not VEGFR as initially stated - this is a key correction)
- **Mechanism of Action**: Direct binding and sequestration of soluble VEGF-A
- **Binding Affinity**: >97% of serum VEGF is bound by bevacizumab
- **Half-life**: 20 days (range 11-50 days)
- **Context**: Applied in angiogenic factor/tumor disease model where VEGF is being added as a soluble factor

## 2. Full Causal Chain Analysis

### Step 1: Direct Molecular Interaction (**CAUSAL**)
Bevacizumab directly binds to circulating VEGF-A protein through high-affinity antibody-antigen interactions, forming stable immune complexes. This prevents VE

In [48]:
structure_explain_df

,index,input_perturbation,thinking,answer,explain,dag,raw_response,question,input_report_text,additional_info
0,0,"{'index': 0, 'perturbation': {'context': {'per...","Let me work through this step by step, based o...",Bevacizumab is a humanized monoclonal antibody...,"set_context(cell_type=""endothelial cells"", dis...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n1""...",<think>\nLet me work through this step by step...,**Q: How does the following perturbation influ...,# Mechanistic Report: Bevacizumab Effects in A...,## KNOWLEDGE GRAPH INFORMATION\nThe retrieved ...
1,1,"{'index': 1, 'perturbation': {'context': {'per...","Looking at this perturbation, we have VEGF bei...",Nintedanib functions as an ATP-competitive inh...,"set_context(cell_type=""endothelial cells"", dis...","edge(""n1"", ""n3"", relation=""causal"")\nedge(""n2""...","<think>\nLooking at this perturbation, we have...",**Q: How does the following perturbation influ...,# Mechanistic Report: Nintedanib Inhibition of...,## KNOWLEDGE GRAPH INFORMATION\nThe retrieved ...
2,2,"{'index': 2, 'perturbation': {'context': {'per...",Let me work through this step by step. We have...,Apatinib acts as an ATP-competitive inhibitor ...,"set_context(cell_type=""endothelial cells"", dis...","edge(""n1"", ""n3"", relation=""causal"")\nedge(""n2""...",<think>\nLet me work through this step by step...,**Q: How does the following perturbation influ...,# Biomedical Reasoning Assistant – Mechanistic...,## KNOWLEDGE GRAPH INFORMATION\nThe retrieved ...


In [50]:
# structured explain
print(structure_explain_df.iloc[0]['explain'])

set_context(cell_type="endothelial cells", disease_context="angiogenic tumor model", prior_perturbation="VEGF soluble factor addition")
binds_to(id="n1", actor="Bevacizumab", target="VEGF-A", affinity=">97% serum binding", via="high-affinity antibody-antigen interaction")
modulates_molecule_activity(id="n2", target="VEGFR-2", direction="down", via="VEGF-A sequestration prevents receptor binding")
modulates_molecule_activity(id="n3", target="VEGFR-1", direction="down", via="VEGF-A sequestration prevents receptor binding")
modulates_pathway_activity(id="n4", pathway="PI3K/AKT signaling", direction="down", via="blocked VEGFR signaling")
modulates_pathway_activity(id="n5", pathway="MAPK/ERK signaling", direction="down", via="blocked VEGFR signaling")
modulates_pathway_activity(id="n6", pathway="PLCγ signaling", direction="down", via="blocked VEGFR signaling")
induces_phenotype(id="n7", source="pathway inhibition", phenotype="increased endothelial cell apoptosis", via="loss of VEGF survival

In [51]:
# paragraph
print(structure_explain_df.iloc[0]['answer'])

Bevacizumab is a humanized monoclonal antibody that directly binds and sequesters circulating VEGF-A protein with high affinity (>97% serum VEGF bound), preventing VEGF-A from engaging its cell surface receptors VEGFR-1, VEGFR-2, and neuropilin co-receptors. This receptor blockade causally inhibits downstream PI3K/AKT, MAPK/ERK, and PLCγ signaling pathways, leading to decreased endothelial cell proliferation, migration, and survival while increasing apoptosis. The cellular dysfunction manifests as reduced expression of angiogenic genes (VEGFA, KDR, ANGPT2, MMP9) and upregulation of hypoxia markers (CA9). These mechanistic changes culminate in vascular regression, normalized vessel architecture, and potent anti-angiogenic effects that rescue the pro-angiogenic phenotype induced by VEGF addition in tumor models.
